# Prepare the Legislator Turns as Numbered Sentences

| Item | Description |
|---|---|
| Goal | Turn every legislator speaking turn in the 2,352 sample hearings into a numbered list of sentences. The stance extraction in T2.2.3 asks the model to cite sentence numbers, so the numbering must be fixed before the model sees the text and must be reproducible. |
| Input | `hearings_sst.parquet` from D4, which carries each hearing's speaking turns with the speaker and roster fields. |
| Output | `member_turns.parquet` in the T2.2 Processed folder: one row per legislator turn with its identifiers, speaker fields, and the cleaned sentence list. T2.2.3, T2.4, T2.5, and T3 read it. |
| Variables | • `is_chair`: the turn belongs to the presiding officer, recorded as `bioGuideId` 0000000 with no surname<br>• `sentences`: the cleaned sentences in order; sentence numbers start at 1<br>• `n_sentences`: their count, 0 when the cleaning rules empty the turn |
| Run | First run 2026-08-22, which wrote the stored table. Rebuilt into a scratch copy on 2026-10-31 and found identical to the stored table; the outputs shown are from that rebuild. The stored August file stays the input of the later steps. |


## Step 1. Import packages


In [1]:
import os
import re
import time
from collections import Counter
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import pysbd


## Step 2. Define file locations


In [2]:
PROJECT_FOLDER = Path.cwd().resolve().parents[2]  # <pipeline>/<section>/<subfolder>/ sits three levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("T - Three Step Text Analysis Pipeline/T2 - Topic Extraction and Stance Analysis/T2.2 - Track 2 - Legislator Stance Extraction")
NOTEBOOK_NAME = "T2.2.1 - Prepare - Legislator Turns as Numbered Sentences"  # the notebook filename without .ipynb
RAW_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Raw"
INTERIM_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Interim"
PROCESSED_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Processed"
INPUT_FOLDER = SCRATCH_FOLDER / "D - Data Collection and Processing/D4 - SST Sample Construction" / "Processed"
FIGURE_FOLDER = PROJECT_FOLDER / "R - Results/figures" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
TABLE_FOLDER = PROJECT_FOLDER / "R - Results/tables" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
MODEL_CACHE = SCRATCH_FOLDER / "cache" / "huggingface"
ENV_FILE = PROJECT_FOLDER / "T - Three Step Text Analysis Pipeline/T2 - Topic Extraction and Stance Analysis" / ".env"
for folder in (INTERIM_FOLDER, PROCESSED_FOLDER):
    folder.mkdir(parents=True, exist_ok=True)


## Step 3. Define the sentence rules

The segmenter is `pysbd`, pinned to one version, because the sentence numbers every stance record points at depend on it. A turn is split paragraph by paragraph, so a sentence never crosses a paragraph boundary. Four cleaning rules then apply, in this order. Cut the turn at the first adjournment or inserted-document marker. Drop the speaker-label echo that opens most turns. Drop bracketed stage directions and all-caps headers. Each rule's removals are counted and printed in Step 5.


In [4]:
PYSBD_VERSION = "0.3.4"  # the sentence numbers every record points at depend on the segmenter, so its version is pinned
assert pysbd.__version__ == PYSBD_VERSION, f"pysbd {pysbd.__version__} found, {PYSBD_VERSION} required"
SEGMENTER = pysbd.Segmenter(language="en", clean=False)
MIN_SENTENCE_CHARS = 15  # shorter fragments stay in the numbering but are counted as suspect evidence

# From the first of these markers on, the text is written material, not speech: the post-adjournment
# appendix rides on the last speaker's turn, and inserted statements sit inside mid-hearing turns.
TRUNCATE_AT = re.compile(r"\[\s*Whereupon|hearing (?:was |is )?adjourned|\[The [^\]]{0,80}follows[:.\]]", re.IGNORECASE)
STAGE_DIRECTION = re.compile(r"\[[^\]]{2,60}\]\.?")  # a sentence that is only "[Laughter.]" or "[Applause.]"


def is_header(sentence):
    """An inline all-caps heading such as "STATEMENT OF MR. DAVID KEATING"."""
    letters = [c for c in sentence if c.isalpha()]
    return len(sentence) >= 20 and len(letters) >= 10 and sum(c.isupper() for c in letters) / len(letters) >= 0.8


def clean_turn_sentences(sentences, surname, honorific):
    """Apply the four cleaning rules to one turn and count what each rule removed."""
    removed = {"truncated": 0, "label_echo": 0, "stage_direction": 0, "header": 0}
    for i, sentence in enumerate(sentences):  # 1. cut at the first adjournment or inserted-document marker
        if TRUNCATE_AT.search(sentence):
            removed["truncated"] = len(sentences) - i
            sentences = sentences[:i]
            break
    if sentences:  # 2. drop the leading speaker-label echo ("Senator Dorgan.")
        first = sentences[0].rstrip(".")
        if len(sentences[0]) <= 40 and ((surname and first.endswith(surname)) or (honorific and first == honorific)):
            removed["label_echo"] = 1
            sentences = sentences[1:]
    kept = []
    for sentence in sentences:  # 3. drop stage directions; 4. drop all-caps headers
        if STAGE_DIRECTION.fullmatch(sentence.strip()):
            removed["stage_direction"] += 1
        elif is_header(sentence):
            removed["header"] += 1
        else:
            kept.append(sentence)
    return kept, removed


def segment_sentences(text):
    """Split one block of text into sentences; line breaks are flattened first because GPO hard-wraps lines."""
    flat = re.sub(r"\s+", " ", text).strip()
    if not flat:
        return []
    return [s.strip() for s in SEGMENTER.segment(flat) if s.strip()]


def segment_turn(text):
    """Split a turn paragraph by paragraph, so a sentence can never cross a paragraph boundary."""
    sentences = []
    for block in re.split(r"\n\s*\n", text or ""):
        sentences.extend(segment_sentences(block))
    return sentences


def format_numbered(sentences, start=1):
    """The sentences as the model sees them: one per line, numbered from `start`."""
    return "\n".join(f"[{i}] {s}" for i, s in enumerate(sentences, start))


## Step 4. Select the legislator turns

A turn counts when its speaker is on the committee roster or is the presiding officer. Senate transcripts record the chair as `The Chairman` with no surname, so the chair is kept by its identifier. Witness turns and unattributed front matter are left out; witness content reaches the analysis through the summaries of Track 1.


In [5]:
CHAIR_ID = "0000000"
sst = pd.read_parquet(INPUT_FOLDER / "hearings_sst.parquet", columns=["packageId", "year", "chamber", "turns"])

rows = []
composition = {"member": 0, "chair": 0, "witness_or_other": 0, "not_spoken": 0}
for hearing in sst.itertuples():
    for turn_idx, turn in enumerate(hearing.turns):
        if turn.get("section") != "spoken":
            composition["not_spoken"] += 1
            continue
        is_chair = turn.get("bioGuideId") == CHAIR_ID
        if not (turn.get("is_member") or is_chair):
            composition["witness_or_other"] += 1
            continue
        composition["chair" if is_chair else "member"] += 1
        rows.append({
            "packageId": hearing.packageId, "year": int(hearing.year), "chamber": hearing.chamber, "turn_idx": turn_idx,
            "bioGuideId": turn.get("bioGuideId"), "surname": turn.get("surname"), "honorific": turn.get("honorific"),
            "is_chair": is_chair, "n_words": int(turn.get("n_words") or 0), "text": turn["text"],
        })

turns = pd.DataFrame(rows)
total_spoken = sum(v for k, v in composition.items() if k != "not_spoken")
print(f"hearings          : {len(sst):,}")
for kind, count in composition.items():
    print(f"  {kind:<17}: {count:>8,}")
print(f"legislator turns  : {len(turns):,} ({len(turns) / total_spoken:.0%} of spoken turns)")
print(f"legislator words  : {turns['n_words'].sum() / 1e6:.1f}M")
print(f"hearings w/ none  : {len(sst) - turns['packageId'].nunique():,}")


hearings          : 2,352
  member           :  197,985
  chair            :   23,326
  witness_or_other :  177,718
  not_spoken       :       12
legislator turns  : 221,311 (55% of spoken turns)
legislator words  : 38.6M
hearings w/ none  : 9


## Step 5. Split each turn into sentences and clean them

The segmenter is pure Python, so the turns are split in parallel batches. The order is kept, so parallel work cannot change a sentence number.


In [6]:
def segment_batch(texts):
    return [segment_turn(t) for t in texts]


n_workers = int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 4))
started = time.time()
texts = turns["text"].tolist()
batches = [texts[i:i + 2000] for i in range(0, len(texts), 2000)]
with ProcessPoolExecutor(max_workers=n_workers) as pool:
    segmented = [s for batch in pool.map(segment_batch, batches) for s in batch]
raw_total = sum(len(s) for s in segmented)
print(f"segmented in {time.time() - started:.0f}s on {n_workers} workers ({raw_total:,} raw sentences)")

cleaned, removal_totals = [], Counter()
for sentences, surname, honorific in zip(segmented, turns["surname"], turns["honorific"]):
    kept, removed = clean_turn_sentences(sentences, surname, honorific)
    cleaned.append(kept)
    removal_totals.update(removed)
turns["sentences"] = cleaned
turns["n_sentences"] = turns["sentences"].str.len()

print("removed by rule:")
for rule, count in removal_totals.most_common():
    print(f"  {rule:<16}: {count:>9,} ({count / raw_total:.1%} of raw)")
n_sent = turns["n_sentences"]
print(f"\nsentences kept    : {n_sent.sum():,} ({n_sent.sum() / raw_total:.0%} of raw)")
print(f"per turn          : median {n_sent.median():.0f}, p90 {n_sent.quantile(0.9):.0f}, max {n_sent.max():,}")
print(f"emptied by cleanup: {(n_sent == 0).sum():,} turns (procedural only; the extraction skips them)")
long_left = sum(1 for ss in turns["sentences"] for s in ss if len(s.split()) > 150)
print(f"sentences over 150 words after paragraph-wise segmentation: {long_left:,}")
short = turns["sentences"].map(lambda ss: sum(len(s) < MIN_SENTENCE_CHARS for s in ss)).sum()
print(f"short fragments (under {MIN_SENTENCE_CHARS} chars): {short:,} ({short / max(n_sent.sum(), 1):.1%} of kept)")


segmented in 162s on 8 workers (2,490,103 raw sentences)


removed by rule:
  truncated       : 1,249,776 (50.2% of raw)
  label_echo      :   219,650 (8.8% of raw)
  header          :    28,666 (1.2% of raw)
  stage_direction :     6,350 (0.3% of raw)

sentences kept    : 985,661 (40% of raw)
per turn          : median 2, p90 9, max 3,831
emptied by cleanup: 1,151 turns (procedural only; the extraction skips them)


sentences over 150 words after paragraph-wise segmentation: 237
short fragments (under 15 chars): 105,310 (10.7% of kept)


## Step 6. Read a sample of segmented turns

Hearing speech is full of abbreviations such as `Mr.`, `H.R. 3684`, and `U.S.` that a naive splitter breaks on. The sample is seeded, so a rerun prints the same turns.


In [7]:
sample = turns.loc[turns["n_sentences"] >= 3].sample(50, random_state=1089)
for row in sample.head(8).itertuples():
    who = f"{row.honorific or ''} {row.surname or ''}".strip()
    print("=" * 78)
    print(f"{row.packageId}  turn {row.turn_idx}  [{who}]")
    print(format_numbered(row.sentences[:8]))
    if len(row.sentences) > 8:
        print(f"  ... {len(row.sentences) - 8} more sentences")
print("=" * 78)
print("(8 of the 50 sampled turns shown)")


CHRG-111shrg66169  turn 84  [Senator Lautenberg]
[1] Thank you very much.
[2] And now like to hear from Joseph Kelly, who's the Acting Chief of New Jersey Transit Police.
[3] Mr. Kelly, welcome.
[4] NJ TRANSIT
CHRG-115hhrg26233  turn 31  [Mr. Bera]
[1] Okay.
[2] And then also, obviously, these are multiyear missions, multiyear strategic planning as we're looking at longer-term goals.
[3] You know, the Chairman brought up the International Space Station and we've committed to funding through 2024.
[4] Obviously, that's one thing we've certainly been meeting with folks, chatting with folks at NASA, as well as others, the potential possibility as more commercial entities get into space, as other institutions see this valuable asset and the academic sector and others, the potential of life after 2024.
[5] And I'd be curious in NASA's long-term planning how are you guys thinking about the ISS in longer-term?
CHRG-107shrg89680  turn 4  [Senator Burns]
[1] Senator Burns [continuing].
[2] So t

## Step 7. Save the table

The raw `text` column is dropped. It stays in `hearings_sst.parquet`, and `(packageId, turn_idx)` joins back to it.


In [8]:
out = turns.drop(columns=["text"]).reset_index(drop=True)
out.to_parquet(PROCESSED_FOLDER / "member_turns.parquet", compression="zstd", index=False)
print(f"wrote member_turns.parquet: {len(out):,} rows, {(PROCESSED_FOLDER / 'member_turns.parquet').stat().st_size / 1e6:.1f} MB")


wrote member_turns.parquet: 221,311 rows, 38.4 MB


## Step 8. Check the saved table

- The key `(packageId, turn_idx)` is unique.
- Sentence counts match the sentence lists.
- Chair turns are present.
- At least 95 percent of hearings contribute a turn.


In [9]:
back = pd.read_parquet(PROCESSED_FOLDER / "member_turns.parquet")
assert not back.duplicated(["packageId", "turn_idx"]).any(), "duplicate turn key"
print(f"1) key (packageId, turn_idx) unique across {len(back):,} rows")
assert (back["sentences"].str.len() == back["n_sentences"]).all()
empty = int((back["n_sentences"] == 0).sum())
print(f"2) sentence counts consistent; {empty} empty turns ({empty / len(back):.2%}), which the extraction skips")
assert back["is_chair"].any(), "no chair turns"
print(f"3) chair turns present: {int(back['is_chair'].sum()):,} ({back['is_chair'].mean():.0%} of legislator turns)")
assert back["packageId"].nunique() >= 0.95 * len(sst), "more than 5% of hearings contribute no legislator turns"
print(f"4) {back['packageId'].nunique():,} of {len(sst):,} hearings contribute turns")
print(f"5) segmenter pinned: pysbd {PYSBD_VERSION}")


1) key (packageId, turn_idx) unique across 221,311 rows
2) sentence counts consistent; 1151 empty turns (0.52%), which the extraction skips
3) chair turns present: 23,326 (11% of legislator turns)
4) 2,343 of 2,352 hearings contribute turns
5) segmenter pinned: pysbd 0.3.4
